In [485]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import scipy.stats as sps
from xgboost import XGBClassifier

In [486]:
df = pd.read_csv('UCI_Credit_Card.csv')
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


ID: ID of each client
LIMIT_BAL: Amount of given credit in NT dollars includes individual and family/supplementary credit
SEX: Gender (1=male, 2=female)
EDUCATION: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)
MARRIAGE: Marital status (1=married, 2=single, 3=others)
AGE: Age in years
PAY_0: Repayment status in September, 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
PAY_2: Repayment status in August, 2005 (scale same as above)
PAY_3: Repayment status in July, 2005 (scale same as above)
PAY_4: Repayment status in June, 2005 (scale same as above)
PAY_5: Repayment status in May, 2005 (scale same as above)
PAY_6: Repayment status in April, 2005 (scale same as above)
BILL_AMT1: Amount of bill statement in September, 2005 (NT dollar)
BILL_AMT2: Amount of bill statement in August, 2005 (NT dollar)
BILL_AMT3: Amount of bill statement in July, 2005 (NT dollar)
BILL_AMT4: Amount of bill statement in June, 2005 (NT dollar)
BILL_AMT5: Amount of bill statement in May, 2005 (NT dollar)
BILL_AMT6: Amount of bill statement in April, 2005 (NT dollar)
PAY_AMT1: Amount of previous payment in September, 2005 (NT dollar)
PAY_AMT2: Amount of previous payment in August, 2005 (NT dollar)
PAY_AMT3: Amount of previous payment in July, 2005 (NT dollar)
PAY_AMT4: Amount of previous payment in June, 2005 (NT dollar)
PAY_AMT5: Amount of previous payment in May, 2005 (NT dollar)
PAY_AMT6: Amount of previous payment in April, 2005 (NT dollar)
default.payment.next.month: Default payment (1=yes, 0=no)


In [487]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   300

In [488]:
print('Все данные есть:')
df.apply(lambda col: col.notna().all(),axis='index')

Все данные есть:


ID                            True
LIMIT_BAL                     True
SEX                           True
EDUCATION                     True
MARRIAGE                      True
AGE                           True
PAY_0                         True
PAY_2                         True
PAY_3                         True
PAY_4                         True
PAY_5                         True
PAY_6                         True
BILL_AMT1                     True
BILL_AMT2                     True
BILL_AMT3                     True
BILL_AMT4                     True
BILL_AMT5                     True
BILL_AMT6                     True
PAY_AMT1                      True
PAY_AMT2                      True
PAY_AMT3                      True
PAY_AMT4                      True
PAY_AMT5                      True
PAY_AMT6                      True
default.payment.next.month    True
dtype: bool

In [489]:
df['default.payment.next.month'].value_counts(normalize=True)

default.payment.next.month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Разберемся с категориальными переменными

In [490]:
# 2-female, 1-male -> 1-female, 0-male
df['SEX'] = df['SEX'].map(
    lambda el: {2: 1, 1: 0}[el]
)
df['SEX'].head()

0    1
1    1
2    1
3    1
4    0
Name: SEX, dtype: int64

In [491]:
# Образование 
# (1=аспирантура, 2=университет, 3=средняя школа, 4=другое, 5=неизвестно, 6=неизвестно)
df['EDUCATION'].value_counts()

EDUCATION
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64

In [492]:
mapping_1 = {1: 'graduate', 2: 'university', 3: 'school'}
df['EDUCATION'] = df['EDUCATION'].map(
    lambda el: mapping_1[el] if el in [1, 2, 3] else 'other_education'
)
df['EDUCATION'].value_counts()

EDUCATION
university         14030
graduate           10585
school              4917
other_education      468
Name: count, dtype: int64

In [493]:
encoded_education = pd.get_dummies(df['EDUCATION'])
df.drop('EDUCATION', axis=1, inplace=True)
df = pd.concat([encoded_education, df], axis=1)
df.head()

,graduate,other_education,school,university,ID,LIMIT_BAL,SEX,MARRIAGE,AGE,PAY_0,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,False,False,False,True,1,20000.0,1,1,24,2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,False,False,False,True,2,120000.0,1,2,26,-1,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,False,False,False,True,3,90000.0,1,2,34,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,False,False,False,True,4,50000.0,1,1,37,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,False,False,False,True,5,50000.0,0,1,57,-1,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [494]:
#БРАК : Семейное положение (1=женат/замужем, 2=холост/не замужем, 3=другое)
df['MARRIAGE'].value_counts()

MARRIAGE
2    15964
1    13659
3      323
0       54
Name: count, dtype: int64

In [495]:
mapping_2 = {1: 'married', 2: 'single'}
df['MARRIAGE'] = df["MARRIAGE"].map(
    lambda el: mapping_2[el] if el in [1, 2] else 'other_marriage'
)
df['MARRIAGE'].value_counts()

MARRIAGE
single            15964
married           13659
other_marriage      377
Name: count, dtype: int64

In [496]:
encoded_marriage = pd.get_dummies(df['MARRIAGE'])
df.drop('MARRIAGE', axis=1, inplace=True)
df = pd.concat([encoded_marriage, df], axis=1)
df.head()

,married,other_marriage,single,graduate,other_education,school,university,ID,LIMIT_BAL,SEX,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,True,False,False,False,False,False,True,1,20000.0,1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,False,False,True,False,False,False,True,2,120000.0,1,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,False,False,True,False,False,False,True,3,90000.0,1,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,True,False,False,False,False,False,True,4,50000.0,1,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,True,False,False,False,False,False,True,5,50000.0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


Таким образом, с помощью One Hot Encoding мы избавились от порядка и превратили качественные переменные в метки 

In [497]:
for education in ['graduate', 'university', 'school', 'other_education']:
    print(education)
    _, default = df.loc[df[education], 'default.payment.next.month'].value_counts(normalize=True).to_numpy()
    print('Доля дефолта =', round(default, 4))

print()
for marriage in ['married', 'single', 'other_marriage']:
    print(marriage)
    _, default = df.loc[df[marriage], 'default.payment.next.month'].value_counts(normalize=True).to_numpy()
    print('Доля дефолта =', round(default, 4))

graduate
Доля дефолта = 0.1923
university
Доля дефолта = 0.2373
school
Доля дефолта = 0.2516
other_education
Доля дефолта = 0.0705

married
Доля дефолта = 0.2347
single
Доля дефолта = 0.2093
other_marriage
Доля дефолта = 0.2361


In [498]:
# Проверка дефолта по полу
df.groupby('SEX')['default.payment.next.month'].value_counts(normalize=True)

SEX  default.payment.next.month
0    0                             0.758328
     1                             0.241672
1    0                             0.792237
     1                             0.207763
Name: proportion, dtype: float64

Проверим количество данных для other_education, чтобы понять, вызван ли такой низкий процент случайностью из-за маленького размера выборки

In [499]:
df.other_education.value_counts(normalize=True)

other_education
False    0.9844
True     0.0156
Name: proportion, dtype: float64

Так и оказалось

Попробуем агрегировать

In [500]:
df.columns

Index(['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'ID', 'LIMIT_BAL', 'SEX', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default.payment.next.month'],
      dtype='object')

In [501]:
cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
amount_of_defaults = df[cols].apply(lambda row: row.map(lambda el: el > 0).sum(), axis='columns')


In [502]:
for education in ['graduate', 'university', 'school', 'other_education']:
    print(education)
    default = amount_of_defaults.loc[df[education]].mean()
    print('Кол-во дефолтных месяцев =', round(default, 4))

print()
for marriage in ['married', 'single', 'other_marriage']:
    print(marriage)
    default = amount_of_defaults.loc[df[marriage]].mean()
    print('Кол-во дефолтных месяцев =', round(default, 4))

print()
print(pd.concat([df['SEX'], pd.Series(amount_of_defaults, name='defaults')], axis=1).groupby(
    by='SEX'
).agg({'defaults': 'mean'}))

graduate
Кол-во дефолтных месяцев = 0.6658
university
Кол-во дефолтных месяцев = 0.926
school
Кол-во дефолтных месяцев = 0.9919
other_education
Кол-во дефолтных месяцев = 0.2329

married
Кол-во дефолтных месяцев = 0.8483
single
Кол-во дефолтных месяцев = 0.821
other_marriage
Кол-во дефолтных месяцев = 0.8806

     defaults
SEX          
0    0.918153
1    0.779097


По сути, агрегация в количество дефолтных месяцев с точки зрения влияния демографических показателей не отличается от целевой переменной. Подводя итог: наиболее вероятен дефолт у менее образованных женатых мужчин

Теперь попробуем построить модель. Начнем с логистической регрессии. Предварительно необходимо воспользоваться масштабирование числовых значений, поэтому создадим пайплайн. Необходимо определиться с метрикой. Модель, которая всегда предсказывает "нет дефолта" (класс 0), получит accuracy 78%, но будет совершенно бесполезна. Нужны метрики, которые учитывают качество предсказания именно редкого, но важного класса (дефолт, класс 1). Попробуем построить с помощью ROC-AUC

In [503]:
def create_log_reg(X_col_names: list, num_cols: list):
    '''
    X_col_names – список с названиями иксов
    num_cols – список с названиями числовых столбов, которые необходимо масштабировать
    '''
    X = df[X_col_names]
    y = df['default.payment.next.month']
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols)                 # для числовых — масштабирование
    ])
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression())
    ])

    # Задаём сетку параметров (для логистической регрессии они идут с префиксом 'clf__')
    param_grid = {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__penalty': ['l1', 'l2'],
        'clf__solver': ['liblinear'],  # liblinear поддерживает l1 и l2
        'clf__class_weight': [None, 'balanced']
    }

    # GridSearch с 5-кратной кросс-валидацией, метрика — roc_auc
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)

    grid.fit(X, y)

    print("Лучшие параметры:", grid.best_params_)
    print("Лучший ROC-AUC на кросс-валидации:", round(grid.best_score_, 4))
    
    return grid


In [504]:
num_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

X = ['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'LIMIT_BAL', 'SEX', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

log_reg_model = create_log_reg(X, num_cols)


Лучшие параметры: {'clf__C': 10, 'clf__class_weight': None, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Лучший ROC-AUC на кросс-валидации: 0.6597


Вышло так, что модель не очень точна, но уже лучше, чем ничего. Попробуем использовать другие переменные. Попробуем убрать BILL_AMT

In [505]:
num_cols = ['LIMIT_BAL', 'AGE',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

X = ['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'LIMIT_BAL', 'SEX', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
log_reg_model = create_log_reg(X, num_cols)


Лучшие параметры: {'clf__C': 10, 'clf__class_weight': None, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Лучший ROC-AUC на кросс-валидации: 0.6366


Качество модели ухудшилось

In [506]:
num_cols = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

X = ['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'LIMIT_BAL', 'SEX', 'AGE', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

log_reg_model = create_log_reg(X, num_cols)

Лучшие параметры: {'clf__C': 10, 'clf__class_weight': None, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}
Лучший ROC-AUC на кросс-валидации: 0.6597


Качество модели не поменялось, при этом мы убрали 6 иксов. Попробуем вместо них добавить ранее используемое нами количество просроченных месяцев

In [507]:
df['default_months'] = amount_of_defaults
num_cols = ['AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_months']

X = ['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'SEX', 'AGE', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_months']

log_reg_model = create_log_reg(X, num_cols)

Лучшие параметры: {'clf__C': 1, 'clf__class_weight': None, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}
Лучший ROC-AUC на кросс-валидации: 0.7524


Замена всех отдельных PAY на показатель количество просроченных месяцев существенно улучшила модель: ROC AUC вырос с 66 до 75. Попробуем использовать RandomForest

In [508]:
def create_random_forest(X_col_names: list, num_cols):
    X = df[X_col_names]
    y = df['default.payment.next.month']
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols)                 # для числовых — масштабирование
    ])
    pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),  # тот же ColumnTransformer, что и раньше
    ('clf', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ])

    param_dist = {
    'clf__n_estimators': sps.randint(50, 201),
    'clf__max_depth': [5, 10, 15, None],
    'clf__min_samples_split': sps.randint(2, 11),
    'clf__min_samples_leaf': sps.randint(1, 6),
    'clf__min_samples_leaf': [1, 2]
    }

    grid_rf = RandomizedSearchCV(
    pipeline_rf, param_dist, n_iter=20, cv=5,
    scoring='roc_auc', n_jobs=-1, random_state=42
    )
    grid_rf.fit(X, y)
    print("Лучшие параметры:", grid_rf.best_params_)
    print("Лучший ROC-AUC на кросс-валидации:", round(grid_rf.best_score_, 4))
    return grid_rf

In [509]:
num_cols = ['AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_months']

X = ['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'SEX', 'AGE', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_months']

random_forest_model = create_random_forest(X, num_cols)

Лучшие параметры: {'clf__max_depth': 10, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 3, 'clf__n_estimators': 133}
Лучший ROC-AUC на кросс-валидации: 0.7628


Модель улучшилась чуть более чем на 1%. Попробуем использовать градиентный бустинг

In [510]:
def create_xgb(X_col_names, num_cols):
    X = df[X_col_names]
    y = df['default.payment.next.month']
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols)                 # для числовых — масштабирование
    ])
    pipeline_xgb = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', XGBClassifier(random_state=42, eval_metric='logloss'))
    ])

    param_grid_xgb = {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [3, 5, 7],
        'clf__learning_rate': [0.01, 0.1],
        'clf__subsample': [0.8, 1.0],
        'clf__colsample_bytree': [0.8, 1.0]
    }

    grid_xgb = GridSearchCV(pipeline_xgb, param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_xgb.fit(X, y)
    print("Лучшие параметры:", grid_xgb.best_params_)
    print("Лучший ROC-AUC на кросс-валидации:", round(grid_xgb.best_score_, 4))
    return grid_xgb

In [511]:
num_cols = ['AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
            'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_months']

X = ['married', 'other_marriage', 'single', 'graduate', 'other_education',
       'school', 'university', 'SEX', 'AGE', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default_months']

xgb_model = create_xgb(X, num_cols)

Лучшие параметры: {'clf__colsample_bytree': 1.0, 'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 100, 'clf__subsample': 0.8}
Лучший ROC-AUC на кросс-валидации: 0.7658


Таким образом, у нас получилось построить модель с ROC-AUC 0,7658. Лучшей моделью получился градиентный бустинг с параметрами {'clf__colsample_bytree': 1.0, 'clf__learning_rate': 0.1, 'clf__max_depth': 3, 'clf__n_estimators': 100, 'clf__subsample': 0.8}